In [1]:
import os
import sys
import pyspark
print(pyspark.__path__[0])
#os.environ["SPARK_HOME"] = pyspark.__path__[0]
print(os.environ.get("SPARK_HOME"))

/usr/local/spark/python/pyspark
/usr/local/spark


In [2]:
from pyspark.sql import SparkSession
import delta
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType, ArrayType
)
import pyspark

#os.environ["SPARK_HOME"] = pyspark.__path__[0]
os.environ["SPARK_HOME"] = os.environ.get("SPARK_HOME")
#os.environ["SPARK_HOME"] = "/usr/local/spark"

# 1. Asegurar que Python apunte al entorno aislado del contenedor actual
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Forzar almacenamiento temporal en el volumen con permisos
os.environ["SPARK_LOCAL_DIRS"] = "/home/jovyan/work"
os.environ["SPARK_JAVA_TRACKER_DIR"] = "/home/jovyan/work"

# Detener rastros de contextos previos por seguridad
try:
    spark.stop()
except:
    pass



In [3]:
# ==========================================
# ESQUEMA 1: TABLA HISTORIAL
# ==========================================
esquema_historial = StructType([
    StructField("id_cliente", IntegerType(), True),       StructField("ingreso_mensual", DoubleType(), True),
    StructField("SCO_ACT", DoubleType(), True),           StructField("id_producto", IntegerType(), True),
    StructField("tipo_producto", StringType(), True),     StructField("monto_credito", DoubleType(), True),
    StructField("plazo_total", DoubleType(), True),       StructField("pago_fijo", DoubleType(), True),
    StructField("fecha_apertura", TimestampType(), True), StructField("prob_mora", DoubleType(), True),
    StructField("mes_vida", IntegerType(), True),         StructField("plazo_remanente", DoubleType(), True),
    StructField("mora", IntegerType(), True),             StructField("Dias_Mora_Mensual", DoubleType(), True), 
    StructField("Ind_Pago_Minimo", DoubleType(), True),   StructField("Ind_Aumento_Linea", DoubleType(), True),
    
    StructField("pago_realizado", DoubleType(), True),    StructField("fecha_corte", DateType(), True),
    StructField("score_evolutivo", DoubleType(), True),   StructField("saldo", DoubleType(), True),    
    StructField("estatus", StringType(), True),

    StructField("Alerta_Estructuracion", IntegerType(), True), StructField("Riesgo_IP_Geo", IntegerType(), True), 
    StructField("Alerta_In_Out", IntegerType(), True),    StructField("anio_mes", StringType(), True),    
    # ----------------------------------------------------
    # METADATOS DE INGESTA Y AUDITORÍA
    # ----------------------------------------------------
    StructField("batch_id", StringType(), True),     # (Texto) Para saber si vino de una corrida base o un re-proceso
    StructField("fecha_envio", TimestampType(), True),  # (Fecha Sistema) Momento exacto en que Spark tocó el dato
    StructField("_corrupt_record", StringType(), True)    # Dead Letter Queue
])


In [4]:

# ==========================================
# ESQUEMA 2: TABLA CLIENTES
# ==========================================
esquema_clientes = StructType([
    StructField("CVE_MUN", StringType(), True),         StructField("CVE_LOC", StringType(), True),
    StructField("Longitud", DoubleType(), True),        StructField("Latitud", DoubleType(), True),
    StructField("CVE_ENT", IntegerType(), True),        StructField("FECHA_ALTA", TimestampType(), True),    
    StructField("ECO_Desarrollo", DoubleType(), True),  StructField("ECO_Dinamismo", DoubleType(), True),
    StructField("ECO_Estabilidad", DoubleType(), True), StructField("ECO_Vulnerabilidad", DoubleType(), True),
    StructField("ECO_Friccion", DoubleType(), True),    StructField("id_cliente", IntegerType(), True),
    StructField("sexo", StringType(), True),            StructField("edad", DoubleType(), True),
    StructField("Fecha_nacimiento", DateType(), True),  StructField("estado_civil", StringType(), True),
    StructField("nivel_edu", StringType(), True),       StructField("numero_de_hijos", DoubleType(), True),
    StructField("NUM_DEP", DoubleType(), True),         StructField("empresa", StringType(), True),
    StructField("Empleo", StringType(), True),          StructField("Telefono", StringType(), True),
    StructField("email", StringType(), True),           StructField("zip", StringType(), True),
    StructField("ANT_LAB_MES", DoubleType(), True),     StructField("ingreso_mensual", DoubleType(), True),
    StructField("gasto_mensual", DoubleType(), True),   StructField("capacidad_ahorro", DoubleType(), True),
    StructField("ING_ANUAL", DoubleType(), True),       StructField("GAS_ANUAL", DoubleType(), True),
    StructField("PATR_EST", DoubleType(), True),        StructField("tiene_auto", StringType(), True),
    StructField("prob_fraude", DoubleType(), True),     StructField("SCO_INI", DoubleType(), True),
    StructField("SCO_ACT", DoubleType(), True),         StructField("NUM_CREDACTI", IntegerType(), True),
    StructField("NUM_CUENTAS", IntegerType(), True),    StructField("NUM_TC", IntegerType(), True),
    StructField("SAL_TOTDEU", DoubleType(), True),      StructField("MONTO_TOTSOL", DoubleType(), True),
    StructField("UTIL_TC", DoubleType(), True),         StructField("MAXDIAS_MORAHIST", DoubleType(), True),
    StructField("plazo_meses", DoubleType(), True),     StructField("TASA_INTASIG", DoubleType(), True),    
 
    StructField("DTI", DoubleType(), True),             StructField("Score_PLD", DoubleType(), True),         
    StructField("Ind_Cuenta_Concentradora", IntegerType(), True), StructField("Ind_Robo_Identidad", IntegerType(), True),     
    StructField("Ind_PEP", DoubleType(), True),         StructField("ID_Contacto_Frecuente", IntegerType(), True),     
     
    
    # ----------------------------------------------------
    # METADATOS DE INGESTA Y AUDITORÍA
    # ----------------------------------------------------
    StructField("batch_id", StringType(), True),     # (Texto) Para saber si vino de una corrida base o un re-proceso
    StructField("fecha_envio", TimestampType(), True),  # (Fecha Sistema) Momento exacto en que Spark tocó el dato
    StructField("_corrupt_record", StringType(), True)    # Dead Letter Queue
])

In [5]:
#import os
#print(os.listdir('/home/jovyan/data'))
#spark.stop()

!sudo chown -R jovyan:users /warehouse

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# ==========================================
# 1. Configuración de la Sesión Maestra
# ==========================================
# Es una sesión ligera (local o conectada a tu clúster) 
# configurada específicamente para habilitar Delta Lake.
spark = SparkSession.builder \
    .appName("Master-Init-Delta-Tables") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Sesión Maestra iniciada. Preparando el entorno (Commit 0)...")

# ==========================================
# 2. Asegurar la Base de Datos
# ==========================================

def init_delta_table(spark_session, db_name, table_name, schema, location):
    # Verificamos si la tabla ya existe en el catálogo
    tables = [r.tableName for r in spark_session.sql(f"SHOW TABLES IN {db_name}").collect()]
    
    if table_name not in tables:
        print(f"Inicializando Commit 0 para {db_name}.{table_name}...")
        
        # Obligamos a Spark a ejecutar la acción (write) con un DF vacío
        df_vacio = spark_session.createDataFrame([], schema)
        df_vacio.write.format("delta").mode("ignore").save(location)
        
        # Registramos la tabla con su esquema en el Metastore
        spark_session.sql(f"""
            CREATE TABLE IF NOT EXISTS {db_name}.{table_name} 
            USING delta 
            LOCATION '{location}'
        """)
        print(f"✅ Tabla {db_name}.{table_name} creada exitosamente.")
    else:
        print(f"✅ La tabla {db_name}.{table_name} ya existe. Omitiendo.")

spark.sql("CREATE DATABASE IF NOT EXISTS pool LOCATION '/warehouse/pool'")

ruta = '/warehouse/pool/historial'
init_delta_table(spark, "pool", "historial", esquema_historial, ruta)

ruta = '/warehouse/pool/clientes'
init_delta_table(spark, "pool", "clientes", esquema_clientes, ruta)

# Apagamos la sesión maestra para liberar recursos
spark.stop()
print("Inicialización terminada. Las sesiones multisesión ya pueden arrancar de forma segura.")

Sesión Maestra iniciada. Preparando el entorno (Commit 0)...
Inicializando Commit 0 para pool.historial...
✅ Tabla pool.historial creada exitosamente.
Inicializando Commit 0 para pool.clientes...
✅ Tabla pool.clientes creada exitosamente.
Inicialización terminada. Las sesiones multisesión ya pueden arrancar de forma segura.


In [7]:

# ## REGRESO A LO SEGURO: El paquete real oficial de la comunidad para tus versiones
# #spark = SparkSession.builder.appName("ReceptorMultiTabla") \
# #    .config( "spark.jars.packages",  
# #             ",".join([ "io.delta:delta-spark_2.13:4.0.1", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1" ]) ) \
# #    .config( "spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension" ) \
# #    .config( "spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog" ) \
# #    .getOrCreate()

# spark = ( SparkSession.builder.appName("ReceptorMultiTabla") \
#     .config( "spark.jars.packages",
#         ",".join([ "io.delta:delta-spark_2.13:4.0.1", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1" ]) ) \
#     .config( "spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension" ) \
#     .config( "spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog" ) \
#     .config( "spark.sql.catalogImplementation", "hive" ) \                ###########################################
#     .config("hive.metastore.uris", "thrift://hive-metastore:9083" ) \     ## Hive
#     .config("spark.sql.warehouse.dir", "/warehouse" ) \                   ###########################################
#     .enableHiveSupport().getOrCreate()
# )


# 1. Configuración de la sesión de Spark
spark = (SparkSession.builder.appName("ReceptorMultiTabla")
    .config("spark.jars.packages", 
            ",".join([ "io.delta:delta-spark_2.12:3.3.2", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3" ]))
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    
    # Directorio base del warehouse (ahora apunta correctamente al HDD físico)
    .config("spark.sql.warehouse.dir", "/warehouse")    
    
    # Optimizaciones HDD
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.default.parallelism", "2")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.HDFSLogStore")
    .config("spark.delta.commitInfo.enabled", "true")
    .config("spark.databricks.delta.merge.repartitionBeforeWrite", "true")
    
    # Control de memoria
    .config("spark.driver.memory", "4g").config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "4").config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.memory.offHeap.enabled", "true").config("spark.sql.memory.offHeap.size", "2g")
    
    # Guardar checkpoints dentro de la estructura del warehouse
    .config("spark.sql.streaming.checkpointLocation", "/warehouse/checkpoints")
    .config("spark.sql.streaming.fileSource.cleaner.enabled", "true")
    .config("spark.sql.streaming.fileSource.cleaner.numDays", "7")    
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.network.timeout", "300s")
    .config("spark.executor.heartbeatInterval", "60s")
    .enableHiveSupport()
    .getOrCreate()
)

# 2. Creación de la base de datos en la ruta unificada del HDD
#spark.sql("CREATE DATABASE IF NOT EXISTS pool LOCATION '/warehouse/pool'")
#spark.sparkContext.setCheckpointDir("/warehouse/checkpoints")


In [8]:
# def table_exists(spark, database, table):
#     try:
#         tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {database}").collect()]
#         return table in tables
#     except Exception:
#         return False

# # Make sure database exists first
# spark.sql("CREATE DATABASE IF NOT EXISTS pool LOCATION '/warehouse/pool'")


# # ==========================================
# # Creación de la tabla CLIENTES e HISTORIAL
# # ==========================================
# if not table_exists(spark, "pool", "clientes"):
    
#     df_vacio_clientes = spark.createDataFrame([], esquema_clientes)# Creamos un DF sin filas pero con tu StructType de clientes
#     df_vacio_clientes.write.format("delta").mode("ignore").save('/warehouse/pool/clientes')# Lo guardamos en Delta. Esto crea el directorio, el Commit 0 y el esquema de golpe
#     spark.sql("CREATE TABLE pool.clientes USING delta LOCATION '/warehouse/pool/clientes'")# Registramos la ruta como tabla en el metastore (ahora sí, ya con estructura)
#     print("Table pool.clientes created with full schema")
# else:
#     print("Table pool.clientes already exists")

# if not table_exists(spark, "pool", "historial"):
#     df_vacio_historial = spark.createDataFrame([], esquema_historial) # Asumiendo que tienes un StructType llamado esquema_historial para tu tabla con mes_vida
#     df_vacio_historial.write.format("delta").mode("ignore").save('/warehouse/pool/historial')
#     spark.sql("CREATE TABLE pool.historial USING delta LOCATION '/warehouse/pool/historial'")
#     print("Table pool.historial created with full schema")
# else:
#     print("Table pool.historial already exists")


In [9]:
# 1. Extracción y parseo del topic "clientes"
df_clientes_raw = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "clientes").option("startingOffsets", "earliest") \
    .option("maxOffsetsPerTrigger", 50000).option("failOnDataLoss", "false").load()
)
# 2. Parseo y detonación del arreglo JSON
df_clientes = (df_clientes_raw.selectExpr("CAST(value AS STRING) as json_payload") 
    .select(from_json(col("json_payload"), ArrayType(esquema_clientes)).alias("data_array")) 
    .select(explode(col("data_array")).alias("data")).select("data.*")
)
# Start streaming
# 6. Iniciar streaming con rutas consistentes y un trigger de prueba (por ejemplo, 15 segundos)
query_clientes = (df_clientes.writeStream.format("delta").outputMode("append") 
    .option("checkpointLocation", "/warehouse/pool/checkpoints/clientes") 
    .option("path", "/warehouse/pool/clientes")
    .trigger(processingTime="2 minutes")#"15 seconds")  # Cambiado temporalmente de 5 min para validar que funcione
    .toTable("pool.clientes")

                 )


# 2. Extracción y parseo del topic "historiales"
df_historiales_raw = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "historial_crediticio").option("startingOffsets", "earliest") \
    .option("maxOffsetsPerTrigger", 100000).option("failOnDataLoss", "false").load()
# 2. Parseo y detonación del arreglo JSON
df_historiales = df_historiales_raw.selectExpr("CAST(value AS STRING) as json_payload") \
    .select(from_json(col("json_payload"), ArrayType(esquema_historial)).alias("data_array")) \
    .select(explode(col("data_array")).alias("data")).select("data.*")
# Now start the stream
query_historiales = (df_historiales.writeStream.format("delta").outputMode("append") 
    .option("checkpointLocation", "/warehouse/pool/checkpoints/historial") 
    .option("path", "/warehouse/pool/historial")
    .trigger(processingTime="2 minutes")
    .toTable("pool.historial")
)




In [10]:
spark.streams.awaitAnyTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [11]:

# Compactar los archivos pequeños en grandes bloques
spark.sql("OPTIMIZE delta.`/warehouse/pool/clientes`")
spark.sql("OPTIMIZE delta.`/warehouse/pool/historial`")

# Apagar la retención de seguridad temporalmente
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# Borrar físicamente los archivos pequeños huérfanos que tengan más de 0 horas de antigüedad
spark.sql("VACUUM delta.`/warehouse/pool/clientes` RETAIN 0 HOURS")
spark.sql("VACUUM delta.`/warehouse/pool/historial` RETAIN 0 HOURS")

#spark.stop()

DataFrame[path: string]

In [ ]:
spark.stop()


In [ ]:



#df_historiales = df_historiales_raw.selectExpr("CAST(value AS STRING) as json_payload") \
#    .select(from_json(col("json_payload"), esquema_historial).alias("data")).select("data.*")


In [ ]:
# # 1. Escritura del flujo de Clientes hacia Delta Lake
# #query_clientes = df_clientes.writeStream.format("delta").outputMode("append") \
# #    .option("checkpointLocation",  "/home/jovyan/data/processed/DataLake/checkpoints/clientes" ) \
# #    .trigger(processingTime="2 minutes") \
# #    .start( "/home/jovyan/data/processed/DataLake/clientes")

# query_clientes = (df_clientes.writeStream.format("delta").outputMode("append") 
#     .option( "checkpointLocation", "/home/jovyan/data/processed/DataLake/checkpoints/clientes" ) 
#     .option( "path", "/home/jovyan/data/processed/DataLake/pool/clientes" )#.option("mergeSchema", "true")
#     .trigger(processingTime="5 minutes").toTable("pool.clientes")
# )









In [ ]:

# # 2. Escritura del flujo de Historiales hacia Delta Lake
# #query_historiales = df_historiales.writeStream.format("delta").outputMode("append") \
# #    .option( "checkpointLocation",  "/home/jovyan/data/processed/DataLake/checkpoints/historial") \
# #    .trigger(processingTime="2 minutes") \
# #    .start( "/home/jovyan/data/processed/DataLake/historial")

# query_historiales = (df_historiales.writeStream.format("delta").outputMode("append") 
#     .option( "checkpointLocation", "/home/jovyan/data/processed/DataLake/checkpoints/historial" ) 
#     .option( "path", "/home/jovyan/data/processed/DataLake/pool/historial" )#.option("mergeSchema", "true") 
#     .trigger(processingTime="5 minutes").toTable("pool.historial")
# )





In [ ]:
# 3. Mantener el clúster escuchando en segundo plano
spark.streams.awaitAnyTermination()

In [ ]:
# Step 1 — check if stream is running
for q in spark.streams.active:
    print(f"Stream: {q.name} | Status: {q.status} | isActive: {q.isActive}")



In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ReceptorMultiTabla") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


In [ ]:


# Compactar los archivos pequeños en grandes bloques
spark.sql("OPTIMIZE delta.`/home/jovyan/data/processed/DataLake/clientes`")
spark.sql("OPTIMIZE delta.`/home/jovyan/data/processed/DataLake/historial`")

# Apagar la retención de seguridad temporalmente
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# Borrar físicamente los archivos pequeños huérfanos que tengan más de 0 horas de antigüedad
spark.sql("VACUUM delta.`/home/jovyan/data/processed/DataLake/clientes` RETAIN 0 HOURS")
spark.sql("VACUUM delta.`/home/jovyan/data/processed/DataLake/historial` RETAIN 0 HOURS")




In [ ]:
# 1. Chequeo de signos vitales: Tabla Clientes
try:
    df_clientes = spark.read.format("delta").load("/home/jovyan/data/processed/DataLake/clientes")
    print("¡La tabla Clientes está viva!")
    df_clientes.show(3)
except Exception as e:
    print("Error en Clientes:", e)

# 2. Chequeo de signos vitales: Tabla Historiales
try:
    df_historiales = spark.read.format("delta").load("/home/jovyan/data/processed/DataLake/historial")
    print("\n¡La tabla Historiales está viva!")
    df_historiales.show(3)
except Exception as e:
    print("Error en Historiales:", e)

In [ ]:
spark = SparkSession.builder \
    .appName("ReceptorMultiTabla") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()



In [ ]:
# 1. Contar registros en la tabla de clientes
Total_clientes = spark.read.format("delta").load("/home/jovyan/data/processed/DataLake/clientes")
Total_clientes.show()
print(f"Total de registros en Clientes: {Total_clientes.count():,}")


In [ ]:
# 2. Contar registros en la tabla de historiales
Total_historiales = spark.read.format("delta").load("/home/jovyan/data/processed/DataLake/historial")
Total_historiales.show()
print(f"Total de registros en Historiales: {Total_historiales.count():,}")



In [ ]:
Python
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# 1. Instanciar la tabla Delta a partir de su ruta en el disco
delta_table = DeltaTable.forPath(spark, "/home/jovyan/data/processed/DataLake/clientes")

# Caso A: Borrado condicional usando una cadena de texto (tipo SQL)
delta_table.delete("id_cliente = '456'")

# Caso B: Borrado condicional usando expresiones de PySpark (útil para limpiar los NULLs)
delta_table.delete(col("id_cliente").isNull())

# Caso C: Borrar ABSOLUTAMENTE TODO el contenido de la tabla (vaciarla)
# delta_table.delete()

In [ ]:
# Borrado condicional mediante Spark SQL
spark.sql("""
    DELETE FROM delta.`/home/jovyan/data/delta/clientes` 
    WHERE id_cliente IS NULL OR _corrupt_record IS NOT NULL
""")

In [ ]:
spark.stop()

In [ ]:
import socket

HOST = "r_streaming"  # o 127.0.0.1 si es mismo contenedor
PORT = 9999

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect((HOST, PORT))

while True:
    data = s.recv(4096)
    if not data:
        break
#    print(data.decode("utf-8", errors="ignore"))


import socket

HOST = "r_streaming"  # o 127.0.0.1 si es mismo contenedor
PORT = 9998

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect((HOST, PORT))

while True:
    data = s.recv(4096)
    if not data:
        break
#    print(data.decode("utf-8", errors="ignore"))




In [ ]:
import shutil

# Borra la carpeta de datos, el historial de Delta (_delta_log) y todo su contenido
shutil.rmtree("/home/jovyan/data/processed/DataLake/clientes")
shutil.rmtree("/home/jovyan/data/processed/DataLake/checkpoints/clientes", ignore_errors=True)
spark.sql("DROP TABLE IF EXISTS clientes")

shutil.rmtree("/home/jovyan/data/processed/DataLake/historial")
shutil.rmtree("/home/jovyan/data/processed/DataLake/checkpoints/historial", ignore_errors=True)
spark.sql("DROP TABLE IF EXISTS historial")


In [ ]:
# Borramos la carpeta de datos y sus checkpoints viejos de la ruta del error
shutil.rmtree("/home/jovyan/data/processed/DataLake/historial", ignore_errors=True)
shutil.rmtree("/home/jovyan/data/processed/DataLake/checkpoints/historial", ignore_errors=True)
print("¡Rutas del DataLake completamente limpias y purgadas!")

In [ ]:
# REGRESO A LO SEGURO: El paquete real oficial de la comunidad para tus versiones
spark = SparkSession.builder.appName("ReceptorMultiTabla") \
    .config( "spark.jars.packages",  
             ",".join([ "io.delta:delta-spark_2.13:4.0.1", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1" ]) ) \
    .config( "spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension" ) \
    .config( "spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog" ) \
    .getOrCreate()
